In [2]:
# ═══ ЯЧЕЙКА 1: Установка, импорт, конфиг ═══
!pip install -q datasets transformers accelerate bitsandbytes sentencepiece protobuf
import re, json, gc, time, random, warnings, os
from datetime import datetime
import pandas as pd, numpy as np, torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42); torch.manual_seed(42)

# === Конфигурация ===
MODELS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
]
SHOTS = [0, 3, 6]
FORMATS = ["word", "json"]
MAX_PER_CLASS = 400
MAX_TEXT = 2500
TEST_SIZE = 50
FINAL_SIZE = 200
SEED = 42
PRICE_PER_GPU_HOUR = 0.20

# Путь к файлу результатов на Google Drive
RESULTS_PATH = "/content/drive/MyDrive/llm_lab_results.json"

import os


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.4 MB/s eta 0:00:00


In [3]:
# ═══ ЯЧЕЙКА 2: Подключение Drive + save/load ═══
from google.colab import drive
drive.mount("/content/drive")

def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"config": {}, "runs": {}, "best_apply": None}

def save_results(data):
    os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def run_key(model, shots, fmt):
    return f"{model}|{shots}|{fmt}"

# Сохраняем конфиг при первом запуске
results = load_results()
results["config"] = {
    "models": MODELS, "shots": SHOTS, "formats": FORMATS,
    "max_per_class": MAX_PER_CLASS, "max_text": MAX_TEXT,
    "test_size": TEST_SIZE, "final_size": FINAL_SIZE, "seed": SEED,
}
save_results(results)
print(f"Файл результатов: {RESULTS_PATH}")
print(f"Уже сохранено прогонов: {len(results['runs'])}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Файл результатов: /content/drive/MyDrive/llm_lab_results.json
Уже сохранено прогонов: 0


In [4]:
# ═══ ЯЧЕЙКА 3: Загрузка данных ═══
TARGET = {
    "030": "Гражданское право", "060": "Труд и занятость",
    "170": "Уголовное право", "180": "Правосудие",
    "020": "Государственное управление", "080": "Финансы",
}
print("Загрузка данных из RusLawOD...")
ds = load_dataset("irlspbru/RusLawOD", split="train", streaming=True)
rows, counts = [], {v: 0 for v in TARGET.values()}
for ex in tqdm(ds, desc="Фильтрация документов", total=100000):
    if len(rows) >= sum([MAX_PER_CLASS] * len(TARGET)): break
    cls = ex.get("classifierByIPS", "")
    if not cls: continue
    m = re.match(r"(\d{3})", cls.split("$")[0].strip())
    if not m or m.group(1) not in TARGET: continue
    label = TARGET[m.group(1)]
    if counts[label] >= MAX_PER_CLASS: continue
    text = (ex.get("textIPS") or "").strip()
    if len(text) < 500: continue
    rows.append({"text": text[:MAX_TEXT], "label": label})
    counts[label] += 1

df = pd.DataFrame(rows)
classes = sorted(df.label.unique())
train, tmp = train_test_split(df, test_size=0.3, stratify=df.label, random_state=SEED)
val, test = train_test_split(tmp, test_size=0.5, stratify=tmp.label, random_state=SEED)
if len(test) > TEST_SIZE * len(classes):
    test = test.groupby("label", group_keys=False).apply(
        lambda x: x.sample(min(len(x), TEST_SIZE), random_state=SEED)
    ).reset_index(drop=True)
print(f"Данные: {len(df)} | Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(df.label.value_counts().to_string())

Загрузка данных из RusLawOD...


README.md:   0%|          | 0.00/6.24k [00:00<?, ?B/s]

Фильтрация документов:   0%|          | 0/100000 [00:00<?, ?it/s]

Данные: 1761 | Train: 1232 | Val: 264 | Test: 265
label
Государственное управление    400
Гражданское право             400
Труд и занятость              400
Финансы                       400
Правосудие                    101
Уголовное право                60


In [5]:
# ═══ ЯЧЕЙКА 4: Функции (модели, промпты, парсинг, метрики) ═══
def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = {"device_map": "auto"}
    if "bnb" not in name.lower() and torch.cuda.is_available():
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(name, **kw)
    model.eval()
    return model, tok

def unload(model, tok):
    del model, tok; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

@torch.no_grad()
def generate(model, tok, prompt, max_tokens=30):
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

def build_prompt(text, n_shots, fmt):
    cs = ", ".join(classes)
    inst = "Ответь только названием отрасли из списка." if fmt == "word" \
        else 'Ответь в формате JSON: {"отрасль": "название"}'
    p = f"Определи отрасль права для юридического текста.\nОтрасли: {cs}.\n{inst}\n\n"
    if n_shots > 0:
        examples = []
        for c in classes:
            pool = train[train.label == c]
            if len(pool): examples.append(pool.iloc[0])
            if len(examples) >= n_shots: break
        for e in examples:
            p += f"Текст: {e.text[:500]}\n"
            p += (f"Ответ: {e.label}\n\n" if fmt == "word"
                  else f'Ответ: {{"отрасль": "{e.label}"}}\n\n')
    p += f"Текст: {text}\nОтвет:"
    return p

def parse(response, fmt):
    r = response.strip()
    if fmt == "word":
        for c in classes:
            if c.lower() in r.lower(): return c
        return None
    try:
        m = re.search(r'\{[^}]*\}', r, re.DOTALL)
        if m:
            data = json.loads(m.group())
            v = str(list(data.values())[0]) if data else ""
            for c in classes:
                if c.lower() in v.lower(): return c
    except: pass
    return None

def calc_metrics(y_true, y_pred):
    valid = [(t, p) for t, p in zip(y_true, y_pred) if p is not None]
    inv = 1 - len(valid) / len(y_true) if y_true else 0
    if not valid: return {"macro_f1": 0, "mcc": 0, "acc": 0, "invalid": 1.0}
    idx = {c: i for i, c in enumerate(classes)}
    yt = [idx[t] for t, _ in valid]
    yp = [idx[p] for _, p in valid]
    return {
        "macro_f1": round(f1_score(yt, yp, average="macro", zero_division=0), 4),
        "mcc": round(matthews_corrcoef(yt, yp), 4),
        "acc": round(accuracy_score(yt, yp), 4),
        "invalid": round(inv, 4),
    }

def run_inference(model, tok, data_df, n_shots, fmt, desc=""):
    preds = []
    for _, row in tqdm(data_df.iterrows(), total=len(data_df), desc=desc, leave=False):
        prompt = build_prompt(row.text, n_shots, fmt)
        try:
            formatted = tok.apply_chat_template(
                [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
        except: formatted = prompt
        try:
            resp = generate(model, tok, formatted)
            preds.append(parse(resp, fmt))
        except: preds.append(None)
    return preds

In [6]:
# ═══ ЯЧЕЙКА 5: Основной цикл сравнения (с пропуском готовых) ═══
results = load_results()

# Генерируем все ожидаемые комбинации
all_combos = [(m, s, f) for m in MODELS for s in SHOTS for f in FORMATS]
pending = [(m, s, f) for m, s, f in all_combos if run_key(m, s, f) not in results["runs"]]

print(f"Всего комбинаций: {len(all_combos)} | Уже готово: {len(all_combos) - len(pending)} | Осталось: {len(pending)}")

for model_name, n_shots, fmt in tqdm(pending, desc="Прогревы"):
    key = run_key(model_name, n_shots, fmt)
    print(f"\n{'='*50}\n{model_name} | {n_shots}-shot | {fmt}\n{'='*50}")

    try:
        model, tok = load_model(model_name)
    except Exception as e:
        print(f"[ERROR] Не удалось загрузить {model_name}: {e}")
        continue

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    preds = run_inference(model, tok, test, n_shots, fmt,
                          desc=f"{model_name.split('/')[-1]} {n_shots}sh {fmt}")
    elapsed = time.time() - t0
    gpu_mem = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0

    m = calc_metrics(test.label.tolist(), preds)
    unload(model, tok)

    results["runs"][key] = {
        "model": model_name, "shots": n_shots, "format": fmt,
        "metrics": m, "time_s": round(elapsed, 1),
        "gpu_mem_gb": round(gpu_mem, 2),
        "cost_per_1000": round(elapsed / len(test) * 1000 / 3600 * PRICE_PER_GPU_HOUR, 4),
        "timestamp": datetime.now().isoformat(),
    }
    save_results(results)  # сохраняем сразу после каждого прогона
    print(f"  F1={m['macro_f1']:.3f} MCC={m['mcc']:.3f} Inv={m['invalid']:.2f} | {elapsed:.0f}s")

print(f"\nВсе сравнения завершены. Итого прогонов: {len(results['runs'])}")

Всего комбинаций: 18 | Уже готово: 0 | Осталось: 18


Прогревы:   0%|          | 0/18 [00:00<?, ?it/s]


Qwen/Qwen2.5-3B-Instruct | 0-shot | word


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

Qwen2.5-0.5B-Instruct 6sh word:   0%|          | 0/265 [00:00<?, ?it/s]

  F1=0.148 MCC=0.029 Inv=0.17 | 237s

Qwen/Qwen2.5-0.5B-Instruct | 6-shot | json


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2.5-0.5B-Instruct 6sh json:   0%|          | 0/265 [00:00<?, ?it/s]

  F1=0.083 MCC=0.019 Inv=0.00 | 316s

Все сравнения завершены. Итого прогонов: 18


In [7]:
# ═══ ЯЧЕЙКА 6: Выбор лучшей + применение к большему срезу ═══
results = load_results()

# Собираем таблицу
rows = []
for key, run in results["runs"].items():
    r = run["metrics"].copy()
    r.update({"model": run["model"], "shots": run["shots"], "format": run["format"],
              "time_s": run["time_s"], "gpu_mem_gb": run["gpu_mem_gb"]})
    rows.append(r)
res_df = pd.DataFrame(rows)
res_df["score"] = res_df.macro_f1 - 0.1 * res_df.invalid
res_df = res_df.sort_values("score", ascending=False).reset_index(drop=True)
print(res_df[["model","shots","format","macro_f1","mcc","acc","invalid","score"]].round(3).to_string(index=False))

best = res_df.iloc[0]
print(f"\nЛучшая: {best.model} | {best.shots}-shot | {best.format} | score={best.score:.3f}")

# Проверяем, нужно ли прогонять применение к большему срезу
apply_key = f"{best.model}|{best.shots}|{best.format}"
if results.get("best_apply") and results["best_apply"].get("key") == apply_key:
    print("Применение к большему срезу уже выполнено, пропускаю.")
    m = results["best_apply"]["metrics"]
    print(f"  F1={m['macro_f1']:.3f} MCC={m['mcc']:.3f} Acc={m['acc']:.3f} Inv={m['invalid']:.3f}")
else:
    final_data = pd.concat([train, val]).sample(min(FINAL_SIZE, len(train)+len(val)), random_state=SEED)
    print(f"\nПрименение лучшей модели к {len(final_data)} документам...")
    model, tok = load_model(best.model)
    preds = run_inference(model, tok, final_data, int(best.shots), best.format,
                          desc="Финальный прогон")
    m = calc_metrics(final_data.label.tolist(), preds)
    unload(model, tok)

    results["best_apply"] = {
        "key": apply_key, "model": best.model, "shots": int(best.shots),
        "format": best.format, "metrics": m,
        "n_samples": len(final_data), "timestamp": datetime.now().isoformat(),
    }
    save_results(results)
    print(f"  F1={m['macro_f1']:.3f} MCC={m['mcc']:.3f} Acc={m['acc']:.3f} Inv={m['invalid']:.3f}")

res_df.to_csv("/content/drive/MyDrive/llm_comparison.csv", index=False)
print("\nСохранено: llm_comparison.csv + llm_lab_results.json на Google Drive")

                     model  shots format  macro_f1    mcc   acc  invalid  score
  Qwen/Qwen2.5-3B-Instruct      6   json     0.342  0.239 0.378    0.011  0.341
  Qwen/Qwen2.5-3B-Instruct      3   word     0.248  0.178 0.328    0.034  0.245
  Qwen/Qwen2.5-3B-Instruct      3   json     0.247  0.211 0.327    0.030  0.244
  Qwen/Qwen2.5-3B-Instruct      0   json     0.238  0.144 0.295    0.015  0.237
Qwen/Qwen2.5-1.5B-Instruct      6   json     0.235  0.152 0.291    0.000  0.235
  Qwen/Qwen2.5-3B-Instruct      0   word     0.203  0.125 0.278    0.008  0.202
  Qwen/Qwen2.5-3B-Instruct      6   word     0.201  0.150 0.327    0.008  0.200
Qwen/Qwen2.5-1.5B-Instruct      6   word     0.190  0.136 0.291    0.000  0.190
Qwen/Qwen2.5-1.5B-Instruct      3   word     0.175  0.142 0.174    0.026  0.172
Qwen/Qwen2.5-1.5B-Instruct      3   json     0.145  0.139 0.158    0.000  0.145
Qwen/Qwen2.5-0.5B-Instruct      6   word     0.148  0.029 0.246    0.170  0.131
Qwen/Qwen2.5-0.5B-Instruct      0   word

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Финальный прогон:   0%|          | 0/200 [00:00<?, ?it/s]

  F1=0.293 MCC=0.179 Acc=0.333 Inv=0.010

Сохранено: llm_comparison.csv + llm_lab_results.json на Google Drive
